- Select the top N years for each topic based on the highest average annual topic score.
- Select representative documents from each year based on the highest document-level topic scores.
- Examine the content of the documents to review the policy context

In [1]:
# Import libraries
import pandas as pd

In [2]:
# Load files
dictionary_policy_trend_by_year = pd.read_csv('dictionary_policy_trend_by_year.csv')
df = pd.read_parquet('final_data.parquet')

In [3]:
TOP_N_YEAR = 2

result = []

for col in dictionary_policy_trend_by_year.columns:
    
    if col.endswith('_avg_rate_per_1000_tokens'):
    
        temp = dictionary_policy_trend_by_year.nlargest(TOP_N_YEAR, col)[['year', col]].copy()
        
        temp['policy_category'] = col.replace('_avg_rate_per_1000_tokens', '')
        
        temp = temp.rename(columns={col: 'avg_rate_per_1000_tokens'})
        
        result.append(temp)

top_two_years_by_dictionary = pd.concat(result).reset_index(drop=True)

top_two_years_by_dictionary = top_two_years_by_dictionary[
    ['policy_category', 'year', 'avg_rate_per_1000_tokens']
]

In [4]:
top_two_years_by_dictionary

,policy_category,year,avg_rate_per_1000_tokens
0,accessibility,2010,13.024948
1,accessibility,2014,11.472086
2,affordability,2024,10.765768
3,affordability,2022,10.679676
4,broadband,2010,20.437190
5,broadband,2011,16.138817
6,compliance,2016,24.196286
7,compliance,2015,20.156170
8,competition,2015,10.093666
9,competition,2011,7.913027


In [5]:
df[['dominant_policy_category', 'dominant_policy_category_score']]

,dominant_policy_category,dominant_policy_category_score
0,spectrum_wireless,36.862004
1,none,0.000000
2,national_security,95.890411
3,consumer_privacy,57.803468
4,media_broadcasting,25.641026
...,...,...
3068,media_broadcasting,29.411765
3069,accessibility,6.802721
3070,accessibility,36.585366
3071,media_broadcasting,68.493151


In [6]:
N_DOCS = 10

representative_documents = []

for _, row in top_two_years_by_dictionary.iterrows():
    
    category = row['policy_category']
    year = row['year']
    
    score_col = f'{category}_rate_per_1000_tokens'
    matched_terms_col = f'{category}_matched_terms'
    
    temp = df[df['year'] == year].copy()
    
    temp = temp.nlargest(N_DOCS, score_col)
    
    temp['selected_policy_category'] = category
    temp['selected_year_avg_rate'] = row['avg_rate_per_1000_tokens']
    temp['document_policy_score'] = temp[score_col]
    temp['matched_terms_for_category'] = temp[matched_terms_col]
    
    representative_documents.append(temp)

representative_documents = pd.concat(representative_documents).reset_index(drop=True)

In [7]:
representative_documents['text_excerpt'] = (
    representative_documents['body_text']
    .astype(str)
    .str.replace('\n', ' ', regex=False)
    .str.replace('\r', ' ', regex=False)
)

In [8]:
text_review = representative_documents[
    [
        'selected_policy_category',
        'year',
        'selected_year_avg_rate',
        'document_policy_score',
        'page_title',
        'bureaus',
        'date',
        'matched_terms_for_category',
        'text_excerpt',
        'webpage_url'
    ]
].copy()

text_review['review'] = ''

text_review.to_excel('text_review.xlsx', index=False)